# 🧠 MIND: Hallucination Detection Enhancement (2026)

This notebook implements the **MIND Framework Upgrade** supporting **Llama 3.1 8B** and the **Multi-Layer Hidden State** strategy. 

### 🚀 Project Novelty (Grad Submission)
- **Model Upgrade**: Replaced legacy Llama 2 with **Llama 3.1 8B**.
- **Multi-Layer Latent Dynamics**: Implemented a 40,960-dimensional feature extraction strategy using:
    1. Concatenated Hidden States from 5 layers.
    2. Multi-Layer Sequence Mean-Pooling.
    3. Inter-Layer Deltas (e.g., L32-L24) to capture latent disagreement.
- **Automated Evaluation**: Integrated GPT-4.1-mini Batch API for cost-effective factual labeling.

## 🛠️ Step 1: Setup & Clone
This cell cleans the environment, clones the **vicky-testing branch**, and installs dependencies.

In [ ]:
import os
import shutil

# 1. Clean up old versions if they exist to avoid branch conflicts
repo_name = "Detecting-and-Mitigating-Hallucinations-in-Open-Domain-QA"
if os.path.exists(repo_name):
    print(f"🧹 Removing existing folder: {repo_name}")
    shutil.rmtree(repo_name)

# 2. Clone the vicky-testing branch specifically
!git clone -b vicky-testing https://github.com/vicky16898/Detecting-and-Mitigating-Hallucinations-in-Open-Domain-QA.git

# 3. Verify Branch
%cd {repo_name}
print("\n📍 Current Branch:")
!git branch

## 📦 Step 2: Install Dependencies

In [ ]:
!pip install -r requirements.txt
!python -m spacy download en_core_web_sm

## 🔑 Step 3: Authentication
Add your `HF_TOKEN` and `OPENAI_API_KEY` to the Colab Secrets (the 🔑 icon on the left sidebar).

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    print("✅ Tokens loaded and set to environment")
except Exception as e:
    print(f"⚠️ Warning: Secrets not found in Colab menu. {e}")

## 🧪 Step 4: Train the Hallucination Classifier

In [ ]:
# 1. Generate Training Samples (Wikipedia-based)
!python src/generate_data.py --model_family llama3base --model_type 8 --gpu 0

# 2. Extract Multi-Layer Hidden State Features
!python src/generate_hd.py --model_family llama3base --model_type 8 --strategy multi_layer --gpu 0

# 3. Train the Classifier (MLP)
!python src/train.py --model_name llama3base8b --strategy multi_layer --device cuda:0

## 📊 Step 5: Automated HELM Evaluation

In [ ]:
# 1. Generate Evaluation Responses
!python src/generate_helm_data.py --model_family llama3base --model_type 8 --gpu 0

# 2. Submit for GPT-4 Auto-Labeling (Batch API)
!python src/label_helm_data.py --model_name llama3base8b --mode submit

### ⏳ NOTE: Fetching Results
OpenAI Batch API processing can take time. After the batch completes, run the cell below to fetch labels and get final scores.

In [ ]:
# 3. Fetch Labels
!python src/label_helm_data.py --model_name llama3base8b --mode fetch

# 4. Extract Evaluation Features
!python src/generate_hd_for_helm.py --model_family llama3base --model_type 8 --strategy multi_layer --gpu 0

# 5. Final AUC / Accuracy Metrics
!python src/detection_score.py --strategy multi_layer --gpu 0